## **앙상블 학습과 랜덤 포레스트 연습 문제**
___
- 출처 : 핸즈온 머신러닝 Ch07 앙상블 학습과 랜덤 포레스트 연습문제 2, 7, 8, 9번
- 이론적 지식을 묻는 문제의 경우 텍스트 셀을 추가하여 정답을 적어주세요.

In [10]:
# import libraries
import numpy as np

### **1. 직접 투표와 간접 투표 분류기 사이의 차이점은 무엇일까요?**
___


- 직접 투표(Hard Voting): 각 분류기의 예측을 모아서 가장 많이 선택된 클래스를 예측값으로 정합니다 (다수결 원칙)

- 간접 투표(Soft Voting): 모든 분류기가 예측한 각 클래스의 확률을 평균 내어, 확률이 가장 높은 클래스를 선택합니다. 일반적으로 직접 투표보다 성능이 더 좋으며, 모든 분류기가 확률을 추정할 수 있어야 합니다.

### **2. 그레디언트 부스팅 앙상블이 훈련 데이터에 과대 적합되었다면 학습률을 어떻게 해야 할까요?**
___

학습률을 낮추어야 합니다. 또한, n_estimators를 조절하거나 조기 종료를 사용하여 최적의 트리 개수를 찾아야 합니다.

### **3. [실습] 다음 지시에 따라 투표 기반 분류 모델을 만들어 보세요**
___

#### **STEP 1. MNIST 데이터를 불러들이고, 훈련, 검증, 테스트 데이터로 나누세요.**

In [11]:
# import MNIST dataset
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, as_frame = False)
X, y = mnist["data"], mnist["target"]

In [12]:
# train/valid/test dataset
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=10000, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=10000, random_state=42)

####  **STEP 2. 랜덤 포레스트 분류기, 엑스트라 트리 분류기, SVM 분류기, MLP 분류기를 훈련시키세요.**
- 모델 파라미터는 `n_estimators=100`, `random_state=42`로 설정합니다.

In [13]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier

In [14]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
et_clf = ExtraTreesClassifier(n_estimators=100, random_state=42)
svm_clf = LinearSVC(max_iter=100, tol=20, random_state=42)
mlp_clf = MLPClassifier(random_state=42)

####  **STEP 3-1. 앞에서 훈련시킨 각 모델을 직접 투표 방법을 사용해 앙상블로 연결하고 훈련시킨 후, `score()`메서드를 이용하여 검증 데이터셋에서의 성능을 평가해보세요.**

In [15]:
# model fitting
estimators = [rf_clf, et_clf, svm_clf, mlp_clf]

####  **STEP 3-2. 검증 데이터셋에서 각 분류 모델의 성능을 `score()` 메서드를 이용하여 확인해보고, 가장 성능이 낮은 모델을 제거하여 그 결과를 비교해보세요.**
- Hint : 가장 성능이 낮은 모델을 제거할 때 `del`를 활용해보세요

In [16]:
# 각 분류 모델 학습

for estimator in estimators:
    print(f"{estimator.__class__.__name__} 학습 중...")
    estimator.fit(X_train, y_train)

RandomForestClassifier 학습 중...
ExtraTreesClassifier 학습 중...
LinearSVC 학습 중...
MLPClassifier 학습 중...


In [18]:
# 각 분류 모델의 성능 확인

for estimator in estimators:
    score = estimator.score(X_valid, y_valid)
    print(f"{estimator.__class__.__name__} 검증 점수: {score:.4f}")

RandomForestClassifier 검증 점수: 0.9692
ExtraTreesClassifier 검증 점수: 0.9715
LinearSVC 검증 점수: 0.0997
MLPClassifier 검증 점수: 0.9610


- Q. 어떤 모델의 성능이 가장 낮나요?
- A.

In [19]:
# 가장 성능이 낮은 모델 제거

del estimators[2]

In [20]:
# model fitting
named_estimators = [
    ("rf_clf", estimators[0]),
    ("et_clf", estimators[1]),
    ("mlp_clf", estimators[2]),
]
voting_clf = VotingClassifier(named_estimators, voting="hard")
voting_clf.fit(X_train, y_train)

,estimators,"[('rf_clf', ...), ('et_clf', ...), ...]"
,voting,'hard'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [21]:
# 모델 제거 후 성능 확인
print(f"투표 분류기(Voting) 최종 점수: {voting_clf.score(X_valid, y_valid):.4f}")

# 투표 분류기에 포함된 개별 모델들의 점수 재확인
for name, estimator in voting_clf.named_estimators_.items():
    print(f"{name} 점수: {estimator.score(X_valid, y_valid):.4f}")


투표 분류기(Voting) 최종 점수: 0.9734
rf_clf 점수: 0.0000
et_clf 점수: 0.0000
mlp_clf 점수: 0.0000


### **4. 다음 단계를 따라 앞에서 훈련시킨 분류 모델들을 이용하여 스태킹 앙상블을 구성해보자.**
___

#### **STEP 1. 3번 문제의 각 분류 모델을 실행해서 검증 세트에서 예측을 만들고, 그 결과로 훈련 세트를 만들어 보세요.**

In [22]:
# 검증 세트의 각 샘플에 대해 개별 모델의 예측을 저장할 행렬 생성
# 행: 검증 세트 샘플 수, 열: 개별 모델 수
X_valid_predictions = np.empty((len(X_valid), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    # 각 모델의 예측값을 행렬의 열에 채워넣습니다.
    X_valid_predictions[:, index] = estimator.predict(X_valid)

####  **STEP 2. 새로운 훈련 세트를 이용하여 랜덤 포레스트 분류 모델을 학습시켜 보세요.**

In [23]:
from sklearn.ensemble import RandomForestClassifier

# 블렌더(최종 모델) 정의 및 학습
# 이 모델은 "개별 모델들의 예측 조합"을 보고 정답을 맞히는 법을 배웁니다.
rnd_forest_blender = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
rnd_forest_blender.fit(X_valid_predictions, y_valid)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,True


- 이 랜덤 포레스트 분류 모델이 바로 블렌더에 해당합니다.

####  **STEP 3. 이제 테스트셋에서 스태킹 앙상블 모델을 평가해보세요.**
- 성능 평가 지표로 **정확도**를 이용하세요.

In [24]:
# 각 분류 모델의 예측을 만들어 새로운 데이터셋 생성
X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)

for index, estimator in enumerate(estimators):
    X_test_predictions[:, index] = estimator.predict(X_test)

In [25]:
# 새로운 데이터셋을 이용하여 블렌더로 예측
y_pred = rnd_forest_blender.predict(X_test_predictions)

In [26]:
# model test
from sklearn.metrics import accuracy_score

# 최종 정확도 측정
stacking_accuracy = accuracy_score(y_test, y_pred)
print(f"스태킹 앙상블의 최종 테스트 정확도: {stacking_accuracy:.4f}")

스태킹 앙상블의 최종 테스트 정확도: 0.9669
